In [ ]:
%pip install openai
%pip install langchain_openai

In [34]:
# 라이브러리 로드
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os

In [36]:
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm  import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

In [39]:
# .env 내용 가져오기
load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]

In [40]:
qwen_llm = ChatOllama(model="qwen3.5:4b")
exaone_llm = ChatOllama(model="exaone3.5:2.4b")
watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000
)

In [7]:

base_url = "https://router.huggingface.co/v1"
model = "Qwen/Qwen2.5-7B-Instruct:together"

In [9]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=base_url,
    api_key=hf_token,
)

completion = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[])


#### LangChain에서 모델 생성하기

In [10]:
from langchain_openai import ChatOpenAI

# 허긍페이스 모델 이용하기
hugging_llm = ChatOpenAI(
  model = model,
  api_key=hf_token,
  base_url=base_url
)

response = hugging_llm.invoke("생성형 AI 설명해줘")
print(response.content)

생성형 AI는 인공지능의 한 분야로, 주어진 입력을 기반으로 새로운 데이터를 생성하는 능력을 가지고 있습니다. 이는 텍스트, 이미지, 음성, 비디오 등 다양한 형태의 데이터를 생성할 수 있습니다. 생성형 AI는 주로 딥러닝 기법, 특히Generative Adversarial Networks (GANs)와 Variational Autoencoders (VAEs)와 같은 모델을 사용합니다.

생성형 AI의 주요 특징은 다음과 같습니다:

1. **데이터 생성**: 새로운 데이터를 생성할 수 있습니다. 예를 들어, 텍스트 생성, 이미지 생성, 음성 합성 등이 있습니다.

2. **학습**: 대규모 데이터셋을 통해 학습하여 데이터의 패턴을 이해하고 이를 기반으로 새로운 데이터를 생성합니다.

3. **응용 분야**: 예술, 디자인, 게임, 영화 제작, 의료, 뉴스 기사 생성 등 다양한 분야에서 활용됩니다.

4. **비판적 학습**: GANs와 같은 생성형 모델은 두 개의 네트워크를 사용하여 생성된 데이터의 질을 향상시킵니다. 하나는 생성자, 다른 하나는 판별자로 불리며, 판별자는 생성된 데이터가 실제 데이터와 얼마나 유사한지 평가합니다.

생성형 AI는 기존의 데이터를 기반으로 새로운 데이터를 생성하는 능력 때문에, 창의성과 예측력이 높은 AI 시스템으로 평가받고 있습니다. 그러나 이 기술은 윤리적 문제와 보안 문제도 동반하기 때문에, 사용 시 신중하게 고려해야 합니다.


In [ ]:
# 유료 LLM
watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000
)

# 로컬 LLM
qwen_llm = ChatOllama(model="qwen3.5:4b")
exaone_llm = ChatOllama(model="exaone3.5:2.4b")

### RAG(Retrieval Augmented Generation) : 검색 증강 생성
- 생성형 AI가 답변을 할 때 외부 문서를 검색한 뒤 그 내용을 기반으로 답변
- 할루시네이션을 줄이는 목적
- 환각증상 억제
- 질문 -> 관련 문서 검색 -> 검색 결과를 LLM에게 전달 -> 답변 생성


### Document Loader
- 다양한 형식의 파일 / URL을 LangChain Document 객체로 변환
- Document = page_content + metadata
- Loader
  - PyPDFLoader
  - JSONLoader
  - TextLoader
  - WebBaseLoader : URL 기반으로

In [ ]:
%pip install pypdf beautifulsoup4 youtube-transcript-api langchain-chroma faiss-cpu pdfplumber
# langchain-chroma faiss-cpu: 백터 저장소
# pdfplumber pypdf: pdf용
%pip install langchain_text_splitters

In [15]:

loader = PyPDFLoader("./data/서울대 미대.pdf")
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")
print(f"총 페이지 텍스트: {docs[0].page_content[:200]}")
print(f"메타 데이터: {docs[0].metadata}")

총 페이지 수: 23
총 페이지 텍스트: 학생생활안내 학부과정 
Department of Design, College of Fine Arts
Seoul National University
서울대학교 미술대학 
디자인전공
2024
메타 데이터: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-02-26T13:44:15+09:00', 'moddate': '2024-02-26T13:44:25+09:00', 'trapped': '/False', 'source': './data/서울대 미대.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}


In [18]:
#csv loader
csv_loader = CSVLoader("./data/restaurant_reviews.csv", encoding="utf-8", csv_args={'delimiter':','})
csv_docs = csv_loader.load()

print(f"총 페이지 수: {len(csv_docs)}")
print(f"총 페이지 텍스트: {csv_docs[0].page_content[:200]}")
print(f"메타 데이터: {csv_docs[0].metadata}")

총 페이지 수: 30
총 페이지 텍스트: review: Absolutely loved this place! The pasta was cooked to perfection and the sauce had such a rich, deep flavor. Service was warm and attentive throughout the entire meal.
메타 데이터: {'source': './data/restaurant_reviews.csv', 'row': 0}


In [ ]:
#csv loader
web_loader = WebBaseLoader(web_paths=['https://python.org', 'https://langchain.com'])
web_docs = web_loader.load()

print(f"총 페이지 수: {len(web_docs)}")
print(f"총 페이지 텍스트: {web_docs[0].page_content[:200]}")
print(f"메타 데이터: {web_docs[0].metadata}")

In [20]:
# glob: 지정된 확장자 피알만 선택
dir_loader = DirectoryLoader(path="./data", glob="**/*.pdf",loader_cls=PyPDFLoader, show_progress=True)
dir_docs = dir_loader.load()

print(f"총 페이지 수: {len(dir_docs)}")
print(f"총 페이지 텍스트: {dir_docs[0].page_content[:200]}")
print(f"메타 데이터: {dir_docs[0].metadata}")

100%|██████████| 5/5 [00:26<00:00,  5.21s/it]

총 페이지 수: 201
총 페이지 텍스트: Ver. 2.89
Ver. 2.89
메타 데이터: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Windows)', 'creationdate': '2026-03-12T11:22:44+09:00', 'moddate': '2026-03-12T11:30:50+09:00', 'trapped': '/False', 'source': 'data/제주관광가이드.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1'}


In [33]:
#LangChain YoutubeLoader() 안 됨
video_id = "07EzMbVH3QE"
ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id, languages=['en'])

# for idx, t in enumerate(transcript,1):
#   print(f"{idx}: {t.text}")
#   print(f"{idx}: {t.start}")
#   print(f"{idx}: {t.duration}")

text = " ".join([t.text for t in transcript])
# print(text)

docs = Document(page_content=text,metadata={
  "source": f"https://www.youtube.com/watch?v={video_id}",
  "video_id": video_id
})

print(docs.page_content[:500])
print(docs.metadata)

Let’s get it Look at it Pay attention Should I break this icy heart? Yes, I mean your startled heart  I like it, you just have to say yes  When I call you it’s “Freeze tag”  (Da da da dun dun) My eyes locked on you  After holding my breath  Taking down a notch on the attitude Wait for the right moment and pose Tiger eyes shining in the darkness Approaching while in disguise  Mesmerizing you with crimson words Claw the moment you let your guard down We rise up above Shouting as if to reach the sk
{'source': 'https://www.youtube.com/watch?v=07EzMbVH3QE', 'video_id': '07EzMbVH3QE'}


### RecursiveCharacterTextSplitter
- 일반적인 RAG에서는 주로 사용
- LLM의 Context Window는 제한이 있어 문서를 한 번에 넣을 수 없음
- TextSplitter는 문서를 적절한 크기의 청크(chunk)로 분할해 줌
- chunk_size: 청크 최대 크기, chunk_overlap: 청크 겹치는 부분(문맥 유지)

크기만 정해주면, 

문서를 쪼개고 오버랩 되고 설정 해줘야함
오버랩사이즈:chunk사이즈의 10~20% 사이즈로 잡으라


In [40]:

loader = PyPDFLoader("./data/서울대 미대.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,chunk_overlap=50, length_function=len, separators=['\n\n','\n',' ','']
)

chunk = splitter.split_documents(docs)
print(f"원본 페이지 수: {len(docs)}")
print(f"분할된 청크 수: {len(chunk)}")
print(f"촛 청크 길이: {len(docs[0].page_content)}")
print(f"촛 청크 내용: {docs[0].page_content}")
print(f"메타 데이터: {docs[0].metadata}")

원본 페이지 수: 23
분할된 청크 수: 65
촛 청크 길이: 104
촛 청크 내용: 학생생활안내 학부과정 
Department of Design, College of Fine Arts
Seoul National University
서울대학교 미술대학 
디자인전공
2024
메타 데이터: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-02-26T13:44:15+09:00', 'moddate': '2024-02-26T13:44:25+09:00', 'trapped': '/False', 'source': './data/서울대 미대.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}


In [ ]:
# 사라짐. 복사해서 붙여넣기

## Embedding
- 텍스트를 고차원 벡터로 변환(의미가 비슷할수록 벡터가 가깝다)
- RAG 에서 임베딩은 1)문서 청크 저장 2) 질문 검색 시 사용
- 임베딩모델과 LLM은 별개

In [ ]:
# 임베딩 전용모델 써야함. 문자를 숫자로 바꾸는
ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

# 단일 텍스트 임베딩: embed_query
vector = ollama_embedding.embed_query("파이썬이란?")
print(f"백터 차원 {len(vector)}")
print(f"첫 5개 값 {vector[:5]}")

백터 차원 768
첫 5개 값 [0.070536666, -0.01034915, 0.02434959, -0.007977234, -0.026814526]


In [12]:
texts=["파이썬이란?","자바란 무엇인가?","오늘 날씨는?"]
# 여러 문서 임베딩: embed_documents
vectors = ollama_embedding.embed_documents(texts)
print(f"임베딩 문서 수 {len(vectors)}")



임베딩 문서 수 3


In [1]:
# 벡타 유사도
import numpy as np

# 코사인 유사도 : 가장 많이 쓰인다
def cosine_sim(a,b):
  a,b = np.array(a), np.array(b)
  return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [14]:
vec1 = ollama_embedding.embed_query("파이썬이란?")
vec2 = ollama_embedding.embed_query("python이란?")
vec3 = ollama_embedding.embed_query("오늘 저녁 뭐 먹지?")

print(f"파이썬 vs python: {cosine_sim(vec1,vec2):.3f}")
print(f"파이썬 vs 저녁: {cosine_sim(vec1,vec3):.3f}")

# 코사인 유사도를 통해서 얼마나 관계나 있나 보는거임.
# 거리가 가까울수록 의미가 같은 것
# 1에 가까울수록 의미가 가깝다. 0에 가까울 수 록 연관이 없는 문장이다




파이썬 vs python: 0.760
파이썬 vs 저3: 0.129


## 벡터 저장소
메모리에 해놓은것이다. 나중에 문서에 대한 처리를 할 때 또 임베딩 처리 다 해야한다
벡터값을 저장할 수 있는 백터 저장소가 필요
일반 데이터 저장하는 것이랑은 다름. 베터값을 저장할 수 있어야 한다.

- ChromaDB, FAISS, Milvus, Qdrant, Pinecone


In [ ]:
%pip install chroma

In [19]:
# 1. 문서 로드 및 분할
loader = PyPDFLoader("./data/서울대 미대.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,chunk_overlap=50, length_function=len, separators=['\n\n','\n',' ','']
)
chunk = splitter.split_documents(docs)

# 2. 임베딩
ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

# 3. 벡터 스토어(ChromaDB) 저장
# embedding_fulltext_search_data 
# 경로는 매번 새롭게 다시 하는게 좋음
vectorstore= Chroma.from_documents(
  documents=chunk, embedding=ollama_embedding, persist_directory="./db/chroma_db", collection_name="my_docs"
)


In [28]:
# 4. 검색(Retrieval)
# 질문 -> 백터 로 변경, 유사도 높은 걸로 몇 개 추출할 것인지
# similarity_search: 질문을 백터로 변경

results = vectorstore.similarity_search("근로장학생 신청 절차는?", k=3)
for idx, doc in enumerate(results,1):
  print(f"\n[결과] {idx} 출처: {doc.metadata}")
  print(f"{doc.page_content[:300]}...")

results_score = vectorstore.similarity_search_with_score("수강신청 내역확인 방법은?", k=5)
for doc, score in results_score:
    print(f"유사도: {score:.3f}, {doc.page_content[:20]}")



[결과] 1 출처: {'moddate': '2024-02-26T13:44:25+09:00', 'page': 5, 'creator': 'Adobe InDesign 19.2 (Windows)', 'source': './data/서울대 미대.pdf', 'producer': 'Adobe PDF Library 17.0', 'trapped': '/False', 'creationdate': '2024-02-26T13:44:15+09:00', 'page_label': '6', 'total_pages': 23}
근로장학생을 선발한다. 
• 근로장학생의 업무
1.  자신의 신청 부서에 따라서 다르지만 디자인전공 학과사무실에서는  
    보조업무 및 청소, 디자인, 정리 등을 담당한다.
2.  자신의 근무시간에 정해진 장소에서 근무해야하며 혹시 근무가 어려울 경우  
    미리 근무지 책임자에게 양해를 구하고 대신 근무를 할 인원을 책임지고  
    보충해야 한다. 
3.  근로장학생은 근무한 시간만큼 시간당 수당을 계산하여 매달 장학금을  
    지급받는다.
◦ 근로장학생...

[결과] 2 출처: {'producer': 'Adobe PDF Library 17.0', 'page_label': '6', 'moddate': '2024-02-26T13:44:25+09:00', 'total_pages': 23, 'page': 5, 'source': './data/서울대 미대.pdf', 'creator': 'Adobe InDesign 19.2 (Windows)', 'trapped': '/False', 'creationdate': '2024-02-26T13:44:15+09:00'}
10 11
• 장학금 신청 
장학금 신청기간에 서울대학교 포털(마이스누)를 통해 교내장학금을 신청한다.
(신청 완료한 학생만이 선정 대상).
※ 가계 곤란 장학금을 신청할 경우, 건강 보험 납부확인서 (부모 및 본인)
    부모 납부합계액이 10만원 이하일 경우에만 서류제출, 건강보험증 사본
• 수여

In [22]:
# db확인
data = vectorstore.get()
print(data['documents'][0][:300])
print(data['metadatas'][0])
print(f'청크 개수: {vectorstore._collection.count()}')

학생생활안내 학부과정 
Department of Design, College of Fine Arts
Seoul National University
서울대학교 미술대학 
디자인전공
2024
{'source': './data/서울대 미대.pdf', 'total_pages': 23, 'page_label': '1', 'trapped': '/False', 'moddate': '2024-02-26T13:44:25+09:00', 'creationdate': '2024-02-26T13:44:15+09:00', 'creator': 'Adobe InDesign 19.2 (Windows)', 'producer': 'Adobe PDF Library 17.0', 'page': 0}
청크 개수: 65


#### ChromaDB vs FAISS
|항목|ChromaDB|FAISS|
|---|---|---|
|저장방식|디스크 기반 영구 저장|메모리 기반 중심(저장 기능)|
|검색속도| 빠름 | 매우 빠름|
|메타데이터 필터링|지원|제한적|
|설치|`pip install chromadb`|`pip install faiss-cpu`|
|서버모드|지원|내장형 서버 기능 없음|
|권장용도|운영형 RAG|빠른 실험/프로토타입|

In [37]:
# 3. 백터 스토어(FAISS) 저장
faiss_store= FAISS.from_documents(
  documents=chunk, 
  embedding=ollama_embedding
)

results = faiss_store.similarity_search("근로장학생 신청 절차는?", k=3)
for idx, doc in enumerate(results,1):
  print(f"\n[결과] {idx} 출처: {doc.metadata}")
  print(f"{doc.page_content[:300]}...")

results_score = faiss_store.similarity_search_with_score("수강신청 내역확인 방법은?", k=5)
for doc, score in results_score:
    print(f"유사도: {score:.3f}, {doc.page_content[:20]}")


[결과] 1 출처: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-02-26T13:44:15+09:00', 'moddate': '2024-02-26T13:44:25+09:00', 'trapped': '/False', 'source': './data/서울대 미대.pdf', 'total_pages': 23, 'page': 5, 'page_label': '6'}
근로장학생을 선발한다. 
• 근로장학생의 업무
1.  자신의 신청 부서에 따라서 다르지만 디자인전공 학과사무실에서는  
    보조업무 및 청소, 디자인, 정리 등을 담당한다.
2.  자신의 근무시간에 정해진 장소에서 근무해야하며 혹시 근무가 어려울 경우  
    미리 근무지 책임자에게 양해를 구하고 대신 근무를 할 인원을 책임지고  
    보충해야 한다. 
3.  근로장학생은 근무한 시간만큼 시간당 수당을 계산하여 매달 장학금을  
    지급받는다.
◦ 근로장학생...

[결과] 2 출처: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-02-26T13:44:15+09:00', 'moddate': '2024-02-26T13:44:25+09:00', 'trapped': '/False', 'source': './data/서울대 미대.pdf', 'total_pages': 23, 'page': 5, 'page_label': '6'}
10 11
• 장학금 신청 
장학금 신청기간에 서울대학교 포털(마이스누)를 통해 교내장학금을 신청한다.
(신청 완료한 학생만이 선정 대상).
※ 가계 곤란 장학금을 신청할 경우, 건강 보험 납부확인서 (부모 및 본인)
    부모 납부합계액이 10만원 이하일 경우에만 서류제출, 건강보험증 사본
• 수여

In [ ]:
# 로컬 저장
faiss_store.save_local("./db/faiss_index")

In [ ]:

# 로컬 저장 후 불러오기
chroma_db=Chroma(persist_directory="./db/chroma_db", embedding_function=ollama_embedding,collection_name="my_docs")
faiss_db=FAISS.load_local("./db/faiss_index",embeddings=ollama_embedding,allow_dangerous_deserialization=True)

In [ ]:
# STEP1: 문서로드
loader = PyPDFLoader("./data/Summary of ChatGPTGPT-4 Research.pdf")

# STEP2: 문서분할
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(loader.load())
 


# STEP3: 인덱싱 - 임베딩
ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

# STEP4: 백터스토어(Chroma or FAISS)
vectorstore= Chroma.from_documents(
  documents=chunks, embedding=ollama_embedding, persist_directory="./db/chroma_db", collection_name="research"
)

# STEP5: as_retriever(): Vector Store을 Retriever 형태로 반환하여 LangChain에 연결
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={'k': 2})

# STEP6: RAG 프롬프트 생성
system_prompt = """\
다음 컨텍스트를 참고하여 잘문에 대답하세요
컨텍스트에 없는 내용은 모른다고 답하세요
컨텍스트:
{context}
"""
rag_prompt = ChatPromptTemplate.from_messages([
  ("system", system_prompt),
  ("human", "{question}")
])

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = {
  "context":retriever | format_docs,
  "question": RunnablePassthrough()
}|rag_prompt | watson_llm | StrOutputParser()


answer = rag_chain.invoke("where can i use ChatGTP?")
print(answer)


chunk 수 [Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-05T00:33:07+00:00', 'author': '', 'keywords': '', 'moddate': '2023-04-05T00:33:07+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/Summary of ChatGPTGPT-4 Research.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Summary of ChatGPT/GPT-4 Research\nand Perspective Towards the Future of Large\nLanguage Models\nYiheng Liu ∗1, Tianle Han ∗1, Siyuan Ma 1, Jiayue Zhang 1,\nYuanyuan Yang1, Jiaming Tian 1, Hao He 1, Antong Li 2, Mengshen\nHe1, Zhengliang Liu 3, Zihao Wu 3, Dajiang Zhu 4, Xiang Li 5, Ning\nQiang1, Dingang Shen 6,7,8, Tianming Liu 3, and Bao Ge †1\n1School of Physics and Information Technology, Shaanxi Normal University, Xi’an\n710119 China'), Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX wi

In [ ]:
# 출처 페이지 정보와 답변 함께 반환

def format_docs_with_source(docs):
  result = []
  for doc in docs:
    src = doc.metadata.get("source","알 수 없음")
    page = doc.metadata.get("page","?")
    result.append(f"[출처: {src} p.{page}\n(doc.page_content)]")
  return ("\n\n".join(result))

rag_with_source = RunnableParallel(answer=rag_chain, sources=retriever)

rag_chain = {
  "context":retriever | format_docs_with_source,
  "question": RunnablePassthrough()
}|rag_prompt | watson_llm | StrOutputParser()


result = rag_with_source.invoke("where can i use ChatGTP?")
print("===답변===")
print(result["answer"])
print("===출처===")
for doc in result["sources"]:
  print(f" - {doc.metadata.get("source")} p.{doc.metadata.get("page","")}")


===답변===
I'm sorry for any confusion, but ChatGPT is the AI language model I am based on. You can use ChatGPT at various platforms and applications that have integrated this technology, such as websites, mobile apps, customer support systems, and more. However, I am unable to provide specific websites or applications that use ChatGPT, as this information is not available in my training data.
===출처===
 - ./data/서울대 미대.pdf p.11
 - ./data/서울대 미대.pdf p.11
